In [1]:
import pathlib
import platform
import pickle

import haiku as hk
import jax
import numpy as np
import hvplot.xarray
import panel as pn
import xarray as xr
import cartopy.crs as ccrs

from graphcast import xarray_jax, xarray_tree, checkpoint, typed_graph
from graphcast.casting import Bfloat16Cast
from graphcast.data_utils import extract_inputs_targets_forcings
from graphcast.mesh_connectivity import get_connected_mesh_nodes, mask_mesh
from graphcast.mesh_graph import faces_to_edges, MeshGraph
from graphcast.model import TASK, TaskConfig, ModelConfig, GraphCast
from graphcast.normalization import InputsAndResiduals
from graphcast.mask import MaskedPredictor
from graphcast.ocean_mesh_utils import read_mesh
from graphcast.xarray_jax import unwrap_vars, unwrap_data

pn.extension()

jax.config.update("jax_traceback_filtering", 'off')

In [2]:
if platform.node().endswith('leonardo.local'):
    root = pathlib.Path("/leonardo_scratch/large/userexternal/scampane/xcast/")
    resolution = "0p25"
    ocean_mesh_filename = "constant_coarse.msh"
else:
    root = pathlib.Path("../")
    resolution = "1"
    ocean_mesh_filename = "constant_mini.msh"

In [3]:
def get_dashboard(dataset, title="", width=600, height=480, **kwargs):

    dataset = xarray_jax.to_np(dataset)
    dataset_min = dataset.min()
    dataset_max = dataset.max()

    variable_selector = pn.widgets.Select(
        name='Variable',
        options=list(dataset.data_vars))

    if 'level' in dataset.coords:
        level_options = {val: idx for idx, val in enumerate(dataset['level'].to_numpy())}
    else:
        level_options = []

    level_selector = pn.widgets.Select(
        name='Level',
        options=level_options)

    if 'time' in dataset.coords:
        time_options = {val: idx for idx, val in enumerate(dataset['time'].dt.days.to_numpy())}
    else:
        time_options = []
        
    time_selector = pn.widgets.DiscreteSlider(
        name='Day',
        options=time_options)
  
    batch_selector = pn.widgets.Select(
        name='Batch',
        options=[] if 'batch' not in dataset.dims else dataset['batch'].to_numpy().tolist())
    
    @pn.depends(variable_selector.param.value, level_selector.param.value, batch_selector.param.value, time_selector.param.value)
    def display(selected_variable, selected_level, selected_batch, selected_time):
        try:
            da = dataset[selected_variable]
            da_min = dataset_min[selected_variable]
            da_max = dataset_max[selected_variable]
            if 'level' in da.dims:
                da = da.isel(level=selected_level)
                da = da.drop_vars('level')
            if 'time' in da.dims:
                da = da.isel(time=selected_time)
                da = da.drop_vars('time')
            if 'batch' in da.dims:
                da = da.isel(batch=selected_batch)
            return da.hvplot.image("lon", "lat", clim=(da_min, da_max), width=width, height=height, **kwargs)
        except Exception as e:
            # Return an informative message if an error occurs during plotting
            return pn.pane.Markdown(f"### Error generating plot for var={selected_variable}, level={selected_level}, batch={selected_batch}, time={selected_time}: {e}")

    dashboard = pn.Row(
        pn.Column(
            title,
            variable_selector,
            level_selector,
            batch_selector,
            time_selector),
        display)
    
    return dashboard

In [4]:
dataset = xr.open_dataset(root / f"data/dataset/dataset_tres-1d_res-{resolution}_levels-10_arco", engine='zarr')

mask = dataset['glorys_mask'].isel(level=0, drop=True)

dataset = dataset.isel(time=slice(0, 3))
dataset = dataset.expand_dims(dim='batch', axis=0)
dataset = dataset.assign_coords({'log-depth': - np.log(dataset['depth']), 'datetime': dataset['time'].expand_dims(dim='batch', axis=0)})
dataset['time'] = dataset['time'] - dataset['time'][0]

dataset

<xarray.Dataset> Size: 50MB
Dimensions:         (batch: 1, time: 3, lat: 181, lon: 360, level: 10)
Coordinates:
    depth           (level) float32 40B 0.494 5.078 11.4 ... 318.1 643.6 902.3
  * lat             (lat) float32 724B 90.0 89.0 88.0 87.0 ... -88.0 -89.0 -90.0
  * level           (level) int64 80B 0 4 8 12 16 20 24 28 32 34
  * lon             (lon) float32 1kB 0.0 1.0 2.0 3.0 ... 357.0 358.0 359.0
  * time            (time) timedelta64[ns] 24B 0 days 1 days 2 days
    log-depth       (level) float32 40B 0.7052 -1.625 -2.434 ... -6.467 -6.805
    datetime        (batch, time) datetime64[ns] 24B 2000-01-01 ... 2000-01-03
Dimensions without coordinates: batch
Data variables: (12/33)
    10u             (batch, time, lat, lon) float32 782kB -4.042 ... -3.437
    10v             (batch, time, lat, lon) float32 782kB 7.837 ... -0.6813
    2d              (batch, time, lat, lon) float32 782kB 264.3 264.3 ... 239.5
    2t              (batch, time, lat, lon) float32 782kB 265.8 265.8 ... 243.8
    deptho          (batch, lat, lon) float32 261kB 4.155e+03 4.155e+03 ... nan
    dis24           (batch, time, lat, lon) float32 782kB nan nan ... nan nan
    ...              ...
    vsd             (batch, time, lat, lon) float32 782kB nan nan ... nan nan
    vsi             (batch, time, lat, lon) float32 782kB 0.1161 0.1164 ... nan
    waverys_deptho  (batch, lat, lon) float32 261kB nan nan nan ... nan nan nan
    waverys_mask    (batch, lat, lon) bool 65kB False False ... False False
    z               (batch, lat, lon) float32 261kB 1.267 1.267 ... 2.705e+04
    zos             (batch, time, lat, lon) float32 782kB -0.622 -0.6224 ... nan
Attributes:
    last_updated:           2025-11-06 01:52:56.448731+00:00
    valid_time_start:       1940-01-01
    valid_time_stop:        2025-04-30
    valid_time_stop_era5t:  2025-10-31

In [5]:
get_dashboard(dataset, title="# Data from ARCO-OCEAN", projection=ccrs.Robinson())

Row
    [0] Column
        [0] Markdown(str)
        [1] Select(name='Variable', options=['10u', '10v', ...], value='10u')
        [2] Select(name='Level', options={np.int64(0): 0, ...}, value=0)
        [3] Select(name='Batch', options=[0], value=0)
        [4] DiscreteSlider(formatter='%d', name='Day', options={np.int64(0): 0, ...}, value=0)
    [1] ParamFunction(function, _pane=HoloViews, defer_load=False)

In [6]:
dataset_jax = xarray_jax.to_jax(dataset)
# Notice: as a side-effect, boolean variables get casted to float32 (which is useful)
dataset_jax = dataset_jax.fillna(value=jax.numpy.float32(0.0))
inputs, targets, forcings = extract_inputs_targets_forcings(dataset=dataset_jax, **TASK, target_lead_times="1d")

In [7]:
inputs

<xarray.Dataset> Size: 26MB
Dimensions:         (batch: 1, time: 2, lat: 181, lon: 360, level: 10)
Coordinates:
    depth           (level) float32 40B 0.494 5.078 11.4 ... 318.1 643.6 902.3
  * level           (level) int64 80B 0 4 8 12 16 20 24 28 32 34
    log-depth       (level) float32 40B 0.7052 -1.625 -2.434 ... -6.467 -6.805
  * lat             (lat) float32 724B 90.0 89.0 88.0 87.0 ... -88.0 -89.0 -90.0
  * lon             (lon) float32 1kB 0.0 1.0 2.0 3.0 ... 357.0 358.0 359.0
  * time            (time) timedelta64[ns] 16B -1 days 00:00:00
Dimensions without coordinates: batch
Data variables: (12/14)
    zos             (batch, time, lat, lon) float32 521kB ...
    mlotst          (batch, time, lat, lon) float32 521kB ...
    thetao          (batch, time, level, lat, lon) float32 5MB ...
    so              (batch, time, level, lat, lon) float32 5MB ...
    uo              (batch, time, level, lat, lon) float32 5MB ...
    vo              (batch, time, level, lat, lon) float32 5MB ...
    ...              ...
    glofas_mask     (batch, lat, lon) float32 261kB ...
    glorys_mask     (batch, level, lat, lon) float32 3MB ...
    lsm             (batch, lat, lon) float32 261kB ...
    uparea          (batch, lat, lon) float32 261kB ...
    waverys_deptho  (batch, lat, lon) float32 261kB ...
    waverys_mask    (batch, lat, lon) float32 261kB ...
Attributes:
    last_updated:           2025-11-06 01:52:56.448731+00:00
    valid_time_start:       1940-01-01
    valid_time_stop:        2025-04-30
    valid_time_stop_era5t:  2025-10-31

In [8]:
targets

<xarray.Dataset> Size: 11MB
Dimensions:    (batch: 1, time: 1, lat: 181, lon: 360, level: 10)
Coordinates:
    depth      (level) float32 40B 0.494 5.078 11.4 21.6 ... 318.1 643.6 902.3
  * level      (level) int64 80B 0 4 8 12 16 20 24 28 32 34
    log-depth  (level) float32 40B 0.7052 -1.625 -2.434 ... -5.762 -6.467 -6.805
  * lat        (lat) float32 724B 90.0 89.0 88.0 87.0 ... -88.0 -89.0 -90.0
  * lon        (lon) float32 1kB 0.0 1.0 2.0 3.0 4.0 ... 356.0 357.0 358.0 359.0
  * time       (time) timedelta64[ns] 8B 1 days
Dimensions without coordinates: batch
Data variables:
    zos        (batch, time, lat, lon) float32 261kB ...
    mlotst     (batch, time, lat, lon) float32 261kB ...
    thetao     (batch, time, level, lat, lon) float32 3MB ...
    so         (batch, time, level, lat, lon) float32 3MB ...
    uo         (batch, time, level, lat, lon) float32 3MB ...
    vo         (batch, time, level, lat, lon) float32 3MB ...
Attributes:
    last_updated:           2025-11-06 01:52:56.448731+00:00
    valid_time_start:       1940-01-01
    valid_time_stop:        2025-04-30
    valid_time_stop_era5t:  2025-10-31

In [9]:
forcings

<xarray.Dataset> Size: 4MB
Dimensions:            (batch: 1, time: 1, lat: 181, lon: 360)
Coordinates:
  * lat                (lat) float32 724B 90.0 89.0 88.0 ... -88.0 -89.0 -90.0
  * lon                (lon) float32 1kB 0.0 1.0 2.0 3.0 ... 357.0 358.0 359.0
  * time               (time) timedelta64[ns] 8B 1 days
Dimensions without coordinates: batch
Data variables: (12/16)
    10u                (batch, time, lat, lon) float32 261kB ...
    10v                (batch, time, lat, lon) float32 261kB ...
    2d                 (batch, time, lat, lon) float32 261kB ...
    2t                 (batch, time, lat, lon) float32 261kB ...
    dis24              (batch, time, lat, lon) float32 261kB ...
    siconc             (batch, time, lat, lon) float32 261kB ...
    ...                 ...
    tp                 (batch, time, lat, lon) float32 261kB ...
    usi                (batch, time, lat, lon) float32 261kB ...
    swh                (batch, time, lat, lon) float32 261kB ...
    vsi                (batch, time, lat, lon) float32 261kB ...
    year_progress_sin  (batch, time) float32 4B 0.02983
    year_progress_cos  (batch, time) float32 4B 0.9996
Attributes:
    last_updated:           2025-11-06 01:52:56.448731+00:00
    valid_time_start:       1940-01-01
    valid_time_stop:        2025-04-30
    valid_time_stop_era5t:  2025-10-31

In [10]:
ocean_mesh, boundary_nodes = read_mesh(root / f"data/geometry/{ocean_mesh_filename}")

ocean_mesh_graph = MeshGraph(vertices=ocean_mesh.vertices, edges=faces_to_edges(ocean_mesh.faces), faces=ocean_mesh.faces)

ocean_mesh_graph

MeshGraph(vertices=array([[  355363.83667211,  -646318.1715593 , -6314105.72949731],
       [  526422.03168573,  -975463.29659177, -6260022.06291247],
       [  821826.65299189,  -979136.71570788, -6227770.99509185],
       ...,
       [  648878.30186558, -6004597.83743799, -2043586.93087766],
       [-3738690.06297659,  1938291.79191839,  4774122.41915513],
       [-3542045.98773108, -2116789.50070507,  4847205.05761867]],
      shape=(6394, 3)), faces=array([[ 356,  351,  354],
       [2333, 4065, 4064],
       [3488, 4476, 4314],
       ...,
       [1222, 1223, 1224],
       [1231, 1229, 1230],
       [1237, 1233, 1236]], shape=(11592, 3)), edges=(array([ 356, 2333, 3488, ..., 1224, 1230, 1236], shape=(34776,)), array([ 351, 4065, 4476, ..., 1222, 1231, 1237], shape=(34776,))))

In [11]:
def get_max_edge_distance(mesh):
  senders, receivers = faces_to_edges(mesh.faces)
  edge_distances = np.linalg.norm(
      mesh.vertices[senders] - mesh.vertices[receivers], axis=-1)
  return edge_distances.max()

query_radius = 0.6 * get_max_edge_distance(ocean_mesh_graph)
query_radius

np.float64(245369.35638309366)

In [12]:
connected_mesh_nodes = get_connected_mesh_nodes(grid_lat=mask['lat'],
                                                grid_lon=mask['lon'],
                                                mesh_graph=ocean_mesh_graph,
                                                grid_mask=mask,
                                                query_radius=query_radius,
                                                workers=-1)

In [13]:
ocean_mesh_graph.vertices.shape[0], len(connected_mesh_nodes)

(6394, 6330)

In [14]:
connected_mesh_graph, _  = mask_mesh(connected_mesh_nodes, ocean_mesh_graph, mode='all')

In [15]:
model_config = ModelConfig(
    latent_size=512,
    gnn_msg_steps=16,
    hidden_layers=1,
    radius_query_fraction_edge_length=0.6)

model_config

ModelConfig(latent_size=512, gnn_msg_steps=16, hidden_layers=1, radius_query_fraction_edge_length=0.6, mesh2grid_edge_normalization_factor=None, per_variable_weights=None)

In [16]:
with (root / 'data/model_config.ckpt').open('wb') as file:
    checkpoint.dump(file, model_config)

In [17]:
task_config = TASK

task_config

TaskConfig(input_variables=('zos', 'mlotst', 'thetao', 'so', 'uo', 'vo', 'deptho', 'z', 'glofas_mask', 'glorys_mask', 'lsm', 'uparea', 'waverys_deptho', 'waverys_mask'), target_variables=('zos', 'mlotst', 'thetao', 'so', 'uo', 'vo'), forcing_variables=('10u', '10v', '2d', '2t', 'dis24', 'siconc', 'sithick', 'sp', 'ssrd', 'strd', 'tp', 'usi', 'swh', 'vsi', 'year_progress_sin', 'year_progress_cos'), levels=(0, 4, 8, 12, 16, 20, 24, 28, 32, 34), input_duration='2d')

In [18]:
with (root / 'task_config.ckpt').open('wb') as file:
    checkpoint.dump(file, task_config)

The location and scale of the variables without `time` dimension (or computed analytically) have been computed as follows.

| Variable name      | Normalization |
|--------------------|---------------|
| `deptho`           | [0, 1]        |
| `waverys_deptho`   | [0, 1]        |
| `tisr`             | [0, 1]        |
| `z`                | [0, 1]        |
| `uparea`           | [0, 1]        |
| `glorys_mask`      | none          |
| `glofas_mask`      | none          |
| `lsm`              | none          |
| `year_progress_sin`| none          |
| `year_progress_cos`| none          |

In [19]:
normalization_artifacts = xr.open_datatree(root / f"data/dataset/dataset_tres-1d_res-{resolution}_levels-10_normalization", engine='zarr')
normalization_artifacts

<xarray.DataTree>
Group: /
│   Dimensions:  (level: 10, lat: 181, lon: 360)
│   Coordinates:
│       depth    (level) float32 40B ...
│     * lat      (lat) float32 724B 90.0 89.0 88.0 87.0 ... -87.0 -88.0 -89.0 -90.0
│     * level    (level) int64 80B 0 4 8 12 16 20 24 28 32 34
│     * lon      (lon) float32 1kB 0.0 1.0 2.0 3.0 4.0 ... 356.0 357.0 358.0 359.0
├── Group: /inputs
│   ├── Group: /inputs/location
│   │       Dimensions:         (lat: 181, lon: 360, level: 10)
│   │       Data variables: (12/33)
│   │           10u             (lat, lon) float32 261kB ...
│   │           10v             (lat, lon) float32 261kB ...
│   │           2d              (lat, lon) float32 261kB ...
│   │           2t              (lat, lon) float32 261kB ...
│   │           deptho          (lat, lon) float32 261kB ...
│   │           dis24           (lat, lon) float32 261kB ...
│   │           ...              ...
│   │           vsd             (lat, lon) float32 261kB ...
│   │           vsi             (lat, lon) float32 261kB ...
│   │           waverys_deptho  (lat, lon) float32 261kB ...
│   │           waverys_mask    (lat, lon) float32 261kB ...
│   │           z               (lat, lon) float32 261kB ...
│   │           zos             (lat, lon) float32 261kB ...
│   │       Attributes:
│   │           last_updated:           2025-11-06 01:52:56.448731+00:00
│   │           valid_time_start:       1940-01-01
│   │           valid_time_stop:        2025-04-30
│   │           valid_time_stop_era5t:  2025-10-31
│   └── Group: /inputs/scale
│           Dimensions:         (lat: 181, lon: 360, level: 10)
│           Data variables: (12/33)
│               10u             (lat, lon) float32 261kB ...
│               10v             (lat, lon) float32 261kB ...
│               2d              (lat, lon) float32 261kB ...
│               2t              (lat, lon) float32 261kB ...
│               deptho          (lat, lon) float32 261kB ...
│               dis24           (lat, lon) float32 261kB ...
│               ...              ...
│               vsd             (lat, lon) float32 261kB ...
│               vsi             (lat, lon) float32 261kB ...
│               waverys_deptho  (lat, lon) float32 261kB ...
│               waverys_mask    (lat, lon) float32 261kB ...
│               z               (lat, lon) float32 261kB ...
│               zos             (lat, lon) float32 261kB ...
│           Attributes:
│               last_updated:           2025-11-06 01:52:56.448731+00:00
│               valid_time_start:       1940-01-01
│               valid_time_stop:        2025-04-30
│               valid_time_stop_era5t:  2025-10-31
└── Group: /residuals
    └── Group: /residuals/scale
            Dimensions:  (lat: 181, lon: 360, level: 10)
            Data variables: (12/25)
                10u      (lat, lon) float32 261kB ...
                10v      (lat, lon) float32 261kB ...
                2d       (lat, lon) float32 261kB ...
                2t       (lat, lon) float32 261kB ...
                dis24    (lat, lon) float32 261kB ...
                i10fg    (lat, lon) float32 261kB ...
                ...       ...
                usd      (lat, lon) float32 261kB ...
                usi      (lat, lon) float32 261kB ...
                vo       (level, lat, lon) float32 3MB ...
                vsd      (lat, lon) float32 261kB ...
                vsi      (lat, lon) float32 261kB ...
                zos      (lat, lon) float32 261kB ...

In [20]:
mean_by_level = normalization_artifacts['/inputs/location'].dataset
mean_by_level = mean_by_level.fillna(0.0)

stddev_by_level = normalization_artifacts['/inputs/scale'].dataset
stddev_by_level = stddev_by_level.fillna(1.0)
stddev_by_level = stddev_by_level.clip(min=1e-18)

diffs_stddev_by_level = normalization_artifacts['/residuals/scale'].dataset
diffs_stddev_by_level = diffs_stddev_by_level.fillna(1.0)
diffs_stddev_by_level = diffs_stddev_by_level.clip(min=1e-18)

In [21]:
# TODO: find out how tisr and progress variables are normalized in original GraphCast code

In [22]:
get_dashboard(stddev_by_level, title='# Std by level', projection=ccrs.Robinson())

Row
    [0] Column
        [0] Markdown(str)
        [1] Select(name='Variable', options=['10u', '10v', ...], value='10u')
        [2] Select(name='Level', options={np.int64(0): 0, ...}, value=0)
        [3] Select(name='Batch')
        [4] DiscreteSlider(formatter='%d', name='Day')
    [1] ParamFunction(function, _pane=HoloViews, defer_load=False)

In [23]:
# Deeper one-step predictor.
predictor = GraphCast(model_config, 
                      task_config, 
                      grid_lat=dataset['lat'].to_numpy(), 
                      grid_lon=dataset['lon'].to_numpy(), 
                      grid_mask=mask,
                      mesh_graph=connected_mesh_graph,
                      boundary_nodes=boundary_nodes)

In [24]:
with (root / 'data/model.pickle').resolve().open('wb') as file:
    pickle.dump(predictor, file, pickle.HIGHEST_PROTOCOL)

In [25]:
# Modify inputs/outputs to `graphcast.GraphCast` to handle conversion to from/to float32 to/from BFloat16.
predictor = Bfloat16Cast(predictor)

# Modify inputs/outputs to `casting.Bfloat16Cast` so the casting to/from BFloat16 happens after applying normalization to the inputs/targets.
predictor = InputsAndResiduals(
    predictor,
    diffs_stddev_by_level=diffs_stddev_by_level,
    mean_by_level=mean_by_level,
    stddev_by_level=stddev_by_level)

# Mask inputs/outputs replacing missing values with 0.0
predictor = MaskedPredictor(predictor, mask=dataset['glorys_mask'], value=0.0)

In [26]:
@hk.without_apply_rng
@hk.transform
def run_forward(inputs, targets_template, forcings):
  return predictor(inputs, targets_template=targets_template, forcings=forcings)

run_forward_jit = jax.jit(run_forward.apply)

In [27]:
key = jax.random.key(0)

params = run_forward.init(rng=key, inputs=inputs, targets_template=targets, forcings=forcings)

In [28]:
forecast = run_forward_jit(params=params, inputs=inputs, targets_template=targets, forcings=forcings)

In [29]:
get_dashboard(forecast, title="# Forecast", projection=ccrs.Robinson())

Row
    [0] Column
        [0] Markdown(str)
        [1] Select(name='Variable', options=['mlotst', 'so', ...], value='mlotst')
        [2] Select(name='Level', options={np.int64(0): 0, ...}, value=0)
        [3] Select(name='Batch', options=[0], value=0)
        [4] DiscreteSlider(formatter='%d', name='Day', options={np.int64(1): 0}, value=0)
    [1] ParamFunction(function, _pane=HoloViews, defer_load=False)

In [30]:
@hk.without_apply_rng
@hk.transform
def loss_fn(inputs, targets, forcings):
  loss, diagnostics = predictor.loss(inputs, targets, forcings=forcings, levels_normalization_coord='log-depth')
  return xarray_tree.map_structure(
      lambda x: unwrap_data(x.mean(), require_jax=True),
      (loss, diagnostics))

loss_fn_apply_jit = jax.jit(loss_fn.apply)

In [31]:
loss, diagnostics = loss_fn_apply_jit(params, inputs, targets, forcings)
loss

Array(2.703125, dtype=float32)

In [32]:
loss_grad = jax.grad(loss_fn_apply_jit, has_aux=True)

In [33]:
# Needs more than 20GB of RAM!
# loss_grad(params, inputs, targets, forcings)